In [29]:
import numpy as np
from pathlib import Path
import torch
from PIL import Image

from torch.utils.data import Dataset, DataLoader, Subset
from typing import Literal, Any
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, accuracy_score, roc_auc_score
from sklearn.svm import SVC

## 01 A bit of copy-paste from previous classes

In [2]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [3]:
root_path = '/mnt/ml/resources/wb2/CelebA'

In [4]:
torch.manual_seed(123)

In [5]:
class CelebaDataset(Dataset):
    
    def __init__(self, path: str, label: str, group: str, processor) -> None:
        super().__init__()
        
        self.path = Path(path)
        self.label = label
        self.group = group
        self.processor = processor
        
        self.metadata = pd.read_csv(self.path / 'list_attr_celeba.txt', skiprows=1, sep=';')
        self.img_path = self.path / 'img_align_celeba'
        
    def __len__(self) -> int:
        return self.metadata.shape[0]
    
    def __getitem__(self, index) -> Any:
        img_path = self.metadata.iloc[index].filename
        img = Image.open(str(self.img_path / img_path)).convert("RGB")
        inputs = self.processor(images=img, return_tensors="pt")
        inputs['pixel_values'] = inputs['pixel_values'][0, :, :, :]
        
        label = self.metadata.iloc[index][self.label]
        group = self.metadata.iloc[index][self.group]
        
        return inputs, label, group

Manual split to train/test, as CelebA doesn't have official splits

In [ ]:
dataset = CelebaDataset(root_path, "Smiling", "Male", processor)

# making smaller datasets for faster computing - but generally bad practise
sub_idx, _, meta_idx, _ = train_test_split(np.arange(len(dataset)), 
                                           dataset.metadata['Smiling'], 
                                           train_size=640, 
                                           random_state=123, 
                                           shuffle=True, 
                                           stratify=dataset.metadata['Smiling'])

train_idx, test_idx = train_test_split(sub_idx, test_size=0.2, random_state=42, shuffle=True, stratify=meta_idx)

dataset_train = Subset(dataset, train_idx)
dataset_test = Subset(dataset, test_idx)

train_loader = DataLoader(dataset_train, batch_size=64, shuffle=False)
test_loader = DataLoader(dataset_test, batch_size=64, shuffle=False)

## 02 Trianing "typical" model

In [ ]:

from sklearn.linear_model import SGDClassifier

# logistic regression with elasticnet penalty trained with batches
classifier = SGDClassifier(loss="log_loss", penalty="elasticnet", random_state=123)

In [ ]:
with torch.no_grad():
    for _ in range(2):  # 2 epochs so it converges a bit - but more should be applied in real life scenario
        for idx, batch in enumerate(tqdm(train_loader)):
            X, y, group = batch
            X = model.get_image_features(**X)['pooler_output']
            
            X = X.cpu().numpy()
            y = y.cpu().numpy()
            group = group.cpu().numpy()
            
            classifier.partial_fit(X, y, classes=[-1, 1])    

100%|██████████| 8/8 [00:18<00:00,  2.33s/it]


In [9]:

ys = []
ys_pred = []
groups = []

with torch.no_grad():
    for idx, batch in enumerate(tqdm(test_loader)):
        X, y, group = batch
        X = model.get_image_features(**X)['pooler_output']
        
        X = X.cpu().numpy()
        y = y.cpu().numpy()
        group = group.cpu().numpy()
        
        y_pred = classifier.predict(X) 
        
        ys.append(y)
        ys_pred.append(y_pred)   
        groups.append(group)

100%|██████████| 2/2 [00:04<00:00,  2.36s/it]


In [10]:
ys = np.concatenate(ys)
ys_pred = np.concatenate(ys_pred)
groups = np.concatenate(groups)

In [11]:
print(f"On all data:\n  Acc: {accuracy_score(ys, ys_pred)}\n  Recall: {recall_score(ys, ys_pred)}\n  Precision: {precision_score(ys, ys_pred)}")

On all data:
  Acc: 0.8359375
  Recall: 0.7580645161290323
  Precision: 0.8867924528301887


In [12]:
ys_subset = ys[groups == -1]
ys_pred_subset = ys_pred[groups == -1]

print(f"Only Female:\n  Acc: {accuracy_score(ys_subset, ys_pred_subset)}\n"
      f"  Recall: {recall_score(ys_subset, ys_pred_subset)}\n  Precision: {precision_score(ys_subset, ys_pred_subset)}")

Only Female:
  Acc: 0.8571428571428571
  Recall: 0.8
  Precision: 0.9230769230769231


In [13]:
ys_subset = ys[groups == 1]
ys_pred_subset = ys_pred[groups == 1]

print(f"Only Male:\n  Acc: {accuracy_score(ys_subset, ys_pred_subset)}\n"
      f"  Recall: {recall_score(ys_subset, ys_pred_subset)}\n  Precision: {precision_score(ys_subset, ys_pred_subset)}")

Only Male:
  Acc: 0.7954545454545454
  Recall: 0.6470588235294118
  Precision: 0.7857142857142857


Why is that? In CelebA, men don't smile much...

In [24]:
dataset.metadata[dataset.metadata['Male'] == 1]['Smiling'].mean()

-0.199422033777862

Women, on the other hand, smile relatively more...

In [25]:
dataset.metadata[dataset.metadata['Male'] == -1]['Smiling'].mean()

0.08104768755553675

So detecting a gender IS a proxy to determine smiling. And CLIP is probably more precise to detect gender than similing (it can be checked really easily!)

## 03 Making "gender-unaware" classifier

Let's first recompute trainig dataset to get data and create diff mean Steering Vector (SV)

In [14]:
groups = []
Xs = []

with torch.no_grad():
    for idx, batch in enumerate(tqdm(train_loader)):
        X, y, group = batch
        X = model.get_image_features(**X)['pooler_output']
        
        X = X.cpu().numpy()
        group = group.cpu().numpy()
        
        Xs.append(X)
        groups.append(group)

100%|██████████| 8/8 [00:17<00:00,  2.22s/it]


In [15]:
Xs = np.concatenate(Xs)
groups = np.concatenate(groups)

...which is defined as simple as difference of means of groups

In [16]:
sv = Xs[groups == 1].mean(axis=0) - Xs[groups == -1].mean(axis=0)

In [17]:
def orthogonalize(X, v, eps=1e-12):

    denom = np.dot(v, v) + eps  # (v^T v)
    projection_coeffs = X @ v / denom  # shape (N,)

    # subtract projection
    X_orth = X - np.outer(projection_coeffs, v) * 0.99

    return X_orth

Now, we will train the same model, in the same way, but we will project latent onto space orthogonal to the SV during training and testing

In [18]:

classifier_debiased = SGDClassifier(loss="log_loss", penalty="elasticnet", random_state=123)

with torch.no_grad():
    for epoch in range(2):
        for idx, batch in enumerate(tqdm(train_loader)):
            X, y, group = batch
            X = model.get_image_features(**X)['pooler_output']
            
            X = X.cpu().numpy()
            y = y.cpu().numpy()
            group = group.cpu().numpy()
            
            if epoch == 1:
                X = orthogonalize(X, sv)
            
            classifier_debiased.partial_fit(X, y, classes=[-1, 1])

100%|██████████| 8/8 [00:17<00:00,  2.25s/it]


In [19]:
ys = []
ys_pred = []
groups = []

with torch.no_grad():
    for idx, batch in enumerate(tqdm(test_loader)):
        X, y, group = batch
        X = model.get_image_features(**X)['pooler_output']
        
        X = X.cpu().numpy()
        y = y.cpu().numpy()
        group = group.cpu().numpy()
        
        X = orthogonalize(X, sv)
        
        y_pred = classifier_debiased.predict(X) 
        
        ys.append(y)
        ys_pred.append(y_pred)   
        groups.append(group)

100%|██████████| 2/2 [00:04<00:00,  2.22s/it]


In [20]:
ys = np.concatenate(ys)
ys_pred = np.concatenate(ys_pred)
groups = np.concatenate(groups)

In [21]:
print(f"On all data:\n  Acc: {accuracy_score(ys, ys_pred)}\n  Recall: {recall_score(ys, ys_pred)}\n  Precision: {precision_score(ys, ys_pred)}")

On all data:
  Acc: 0.8984375
  Recall: 0.8870967741935484
  Precision: 0.9016393442622951


In [22]:
ys_subset = ys[groups == -1]
ys_pred_subset = ys_pred[groups == -1]

print(f"Only Female:\n  Acc: {accuracy_score(ys_subset, ys_pred_subset)}\n"
      f"  Recall: {recall_score(ys_subset, ys_pred_subset)}\n  Precision: {precision_score(ys_subset, ys_pred_subset)}")

Only Female:
  Acc: 0.8928571428571429
  Recall: 0.8888888888888888
  Precision: 0.9090909090909091


In [23]:
ys_subset = ys[groups == 1]
ys_pred_subset = ys_pred[groups == 1]

print(f"Only Male:\n  Acc: {accuracy_score(ys_subset, ys_pred_subset)}\n"
      f"  Recall: {recall_score(ys_subset, ys_pred_subset)}\n  Precision: {precision_score(ys_subset, ys_pred_subset)}")

Only Male:
  Acc: 0.9090909090909091
  Recall: 0.8823529411764706
  Precision: 0.8823529411764706


## 04 General considerations / Next steps

We shall still be aware that:

* improvement of metrics and "expected" behaviour don't imply the model can't detect the geneder anymore - probably it still can, it's just harder (especially in linear classification)
* provided example uses minial setup (small data, few epochs) and can be easily recreated with different seed and contradictory results - but the presented behaviour holds in large scale experiments (mostly)
* the presented bias was found in literature - but can and should be re-validated before experiments; the most basic approaches are:
  * check P(label|concept) <- if differs significantly for different labels, then probably it will occur as a bias
  * check group-based metrics <- in our case, we checked Male/Female metrics differ

## 05 Is linear even reasonable?

In [26]:
def cosine_similarity(v, M):
    v = v.reshape(1, -1)

    dot = M @ v.T
    v_norm = np.linalg.norm(v)
    M_norm = np.linalg.norm(M, axis=1, keepdims=True)

    cos = dot / (M_norm * v_norm)
    return cos.squeeze()

In [30]:
nonlinear_classfier = SVC(kernel='rbf', random_state=123)

In [31]:
groups = []
Xs = []

with torch.no_grad():
    for idx, batch in enumerate(tqdm(train_loader)):
        X, y, group = batch
        X = model.get_image_features(**X)['pooler_output']
        
        X = X.cpu().numpy()
        group = group.cpu().numpy()
        
        Xs.append(X)
        groups.append(group)

100%|██████████| 8/8 [00:18<00:00,  2.28s/it]


In [32]:
Xs = np.concatenate(Xs)
groups = np.concatenate(groups)

In [33]:
nonlinear_classfier.fit(Xs, groups)

SVC(random_state=123)

In [36]:
ys = []
ys_pred_sv = []
ys_pred_svm = []
groups = []

with torch.no_grad():
    for idx, batch in enumerate(tqdm(test_loader)):
        X, y, group = batch
        X = model.get_image_features(**X)['pooler_output']
        
        X = X.cpu().numpy()
        y = y.cpu().numpy()
        group = group.cpu().numpy()
        
        X = orthogonalize(X, sv)
        
        y_pred_sv = cosine_similarity(sv, X) 
        y_pred_svm = nonlinear_classfier.predict(X) 
        
        ys.append(y)
        ys_pred_sv.append(y_pred)   
        ys_pred_svm.append(y_pred_svm)
        groups.append(group)

100%|██████████| 2/2 [00:04<00:00,  2.18s/it]


In [37]:
ys = np.concatenate(ys)
ys_pred_sv = np.concatenate(ys_pred_sv)
ys_pred_svm = np.concatenate(ys_pred_svm)
groups = np.concatenate(groups)

In [42]:
print(f"SV:\n  ROCAUC: {roc_auc_score(groups, ys_pred_sv)}")
print(f"SVM:\n  ROCAUC: {roc_auc_score(groups, ys_pred_svm)}")

SV:
  ROCAUC: 0.7884199134199135
SVM:
  ROCAUC: 0.5059523809523809


But formally, benchmarks suggest that SoTA nonlinear approaches are mostly comparable to SoTA linear approaches in this task